# Notebook 03: Measurement and Probabilities

Prerequisite: read Chapter 03 first for probability and shot-count ideas.

This lab is practice-first. You will study sampling variance and confidence after the chapter concept is clear.

## Cell-by-cell Guide
Read this first so every cell is easy to follow.
1. Understand concept and expected result - Notebook 03: Measurement and Probabilities
2. Understand concept and expected result - Real-World Scenario
3. Run logic and inspect output - from qiskit import QuantumCircuit
4. Run logic and inspect output - qc = QuantumCircuit(1, 1)
5. Run logic and inspect output - summary = df.groupby("shots").agg(
6. Run logic and inspect output - best = summary.sort_values("std_p1").iloc[0]

## Real-World Scenario
**Scenario:** Academic Lab Reproducibility Audit

**Problem:** A lab must show that results are repeatable, not lucky one-off outcomes.
**Baseline:** Use fixed shots and repeated runs with summary variance reporting.
**Metric to watch:** Mean probability, spread, and confidence interval width.
**Practical takeaway:** Single-run claims are weak; repeated evidence builds trust.

## Imports and Purpose
This lab uses `QuantumCircuit` to define the experiment, `AerSimulator` to generate repeated shots, `pandas` to collect results, and `math` for the confidence interval calculation.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
import pandas as pd
import math

sim = AerSimulator()

def ci95(p, n):
    z = 1.96
    margin = z * math.sqrt((p * (1 - p)) / n)
    return max(0, p - margin), min(1, p + margin)

In [ ]:
qc = QuantumCircuit(1, 1)
qc.h(0)
qc.measure(0, 0)

rows = []
for shots in [128, 512, 2048]:
    for repeat in range(1, 6):
        counts = sim.run(qc, shots=shots).result().get_counts()
        c1 = counts.get("1", 0)
        p1 = c1 / shots
        lo, hi = ci95(p1, shots)
        rows.append({
            "shots": shots,
            "repeat": repeat,
            "p1": p1,
            "ci95_low": lo,
            "ci95_high": hi
        })

df = pd.DataFrame(rows)
df.head(10)

In [ ]:
summary = df.groupby("shots").agg(
    mean_p1=("p1", "mean"),
    std_p1=("p1", "std"),
    min_p1=("p1", "min"),
    max_p1=("p1", "max")
).reset_index()
summary

In [ ]:
best = summary.sort_values("std_p1").iloc[0]
print("Most stable shot setting in this run:")
print(best.to_dict())
print("Use this with cost context before final benchmark decisions.")